# File 13 — Image Branch Model Comparison
Compares CNN-only vs hybrid probability-level image branch. No training.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve

OUTPUT_DIR = Path(r'D:\DIABETES\diabetes_pipeline_outputs\13_image_branch_comparison')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# CNN-only results
# Lighting-robust CNN is now the baseline CNN-only model
CNN_METRICS = Path(r'D:\DIABETES\diabetes_pipeline_outputs\06_segmented_test_evaluation_lighting_robust\06_lr_test_metrics_selected_threshold.csv')
CNN_METRICS_OLD = Path(r'D:\DIABETES\diabetes_pipeline_outputs\06_segmented_test_evaluation\06_segmented_test_metrics_selected_threshold.csv')  # older non-CLAHE CNN, kept for reference only
CNN_PROB = Path(r'D:\DIABETES\diabetes_pipeline_outputs\06_segmented_test_evaluation_lighting_robust\06_lr_test_probability_metrics.csv')
CNN_PROB_OLD = Path(r'D:\DIABETES\diabetes_pipeline_outputs\06_segmented_test_evaluation\06_segmented_test_probability_metrics.csv')  # older non-CLAHE CNN
CNN_PREDS = Path(r'D:\DIABETES\diabetes_pipeline_outputs\06_segmented_test_evaluation_lighting_robust\06_lr_test_predictions.csv')
CNN_PREDS_OLD = Path(r'D:\DIABETES\diabetes_pipeline_outputs\06_segmented_test_evaluation\06_segmented_test_predictions.csv')  # older non-CLAHE CNN

# Hybrid results
HYBRID_METRICS = Path(r'D:\DIABETES\diabetes_pipeline_outputs\11_hybrid_probability_fusion\11_hybrid_test_metrics_selected_threshold.csv')
HYBRID_PROB = Path(r'D:\DIABETES\diabetes_pipeline_outputs\11_hybrid_probability_fusion\11_hybrid_test_probability_metrics.csv')
HYBRID_PREDS = Path(r'D:\DIABETES\diabetes_pipeline_outputs\11_hybrid_probability_fusion\11_hybrid_test_predictions.csv')
HYBRID_CI = Path(r'D:\DIABETES\diabetes_pipeline_outputs\11_hybrid_probability_fusion\11_hybrid_bootstrap_confidence_intervals.csv')

print('Paths configured.')


Paths configured.


## Load Results


In [2]:
models = {}

for name, m_path, p_path, pred_path in [
    ('LR-CNN-only (lighting-robust)', CNN_METRICS, CNN_PROB, CNN_PREDS),
    ('Hybrid-LR', HYBRID_METRICS, HYBRID_PROB, HYBRID_PREDS),
]:
    if m_path.exists() and p_path.exists():
        m = pd.read_csv(m_path).iloc[0].to_dict()
        p = pd.read_csv(p_path).iloc[0].to_dict()
        m.update(p)
        models[name] = m
        print(f'{name}: loaded')
    else:
        print(f'{name}: MISSING — {m_path.exists()}, {p_path.exists()}')

if len(models) < 2:
    print('WARNING: cannot compare, need both CNN-only and Hybrid results.')


LR-CNN-only (lighting-robust): loaded
Hybrid-LR: loaded


## Comparison Table


In [3]:
COMPARE_METRICS = ['diabetes_recall','fnr','specificity','ppv','npv','f1',
                     'balanced_accuracy','accuracy','brier','roc_auc','pr_auc']

rows = []
for metric in COMPARE_METRICS:
    row = {'metric': metric}
    for model_name, m in models.items():
        row[model_name] = m.get(metric, 'N/A')
    if len(models) == 2:
        vals = list(models.values())
        v0 = vals[0].get(metric); v1 = vals[1].get(metric)
        if isinstance(v0, (int, float)) and isinstance(v1, (int, float)):
            row['diff'] = round(v1 - v0, 4)
        else:
            row['diff'] = 'N/A'
    rows.append(row)

df_comp = pd.DataFrame(rows)
df_comp.to_csv(OUTPUT_DIR / '13_model_comparison_table.csv', index=False)
print(df_comp.to_string(index=False))


           metric  LR-CNN-only (lighting-robust)  Hybrid-LR    diff
  diabetes_recall                       0.920930   0.958140  0.0372
              fnr                       0.079070   0.041860 -0.0372
      specificity                       0.883249   0.868020 -0.0152
              ppv                       0.895928   0.887931 -0.0080
              npv                       0.910995   0.950000  0.0390
               f1                       0.908257   0.921700  0.0134
balanced_accuracy                       0.902089   0.913080  0.0110
         accuracy                       0.902913   0.915049  0.0121
            brier                       0.081027   0.062124 -0.0189
          roc_auc                       0.960076   0.970228  0.0102
           pr_auc                       0.962433   0.969894  0.0075


## Plots


In [4]:
# Metric comparison bar plot
plot_metrics = ['diabetes_recall','specificity','ppv','npv','f1','balanced_accuracy','roc_auc','pr_auc']
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(plot_metrics))
width = 0.35
for i, (model_name, m) in enumerate(models.items()):
    vals = [m.get(pm, 0) for pm in plot_metrics]
    ax.bar(x + i * width, vals, width, label=model_name)
ax.set_xticks(x + width / 2)
ax.set_xticklabels(plot_metrics, rotation=45, ha='right')
ax.set_ylim(0, 1.1)
ax.legend()
ax.set_title('Image Branch Comparison (Baseline: Lighting-Robust CNN-only)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '13_metric_comparison_barplot.png', dpi=100)
plt.close()

# ROC/PR comparison if predictions available
if CNN_PREDS.exists() and HYBRID_PREDS.exists():
    cnn_p = pd.read_csv(CNN_PREDS)
    hyb_p = pd.read_csv(HYBRID_PREDS)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # Determine column names
    cnn_yt = cnn_p['y_true'].values if 'y_true' in cnn_p.columns else None
    cnn_yp = cnn_p['y_prob_diabetes'].values if 'y_prob_diabetes' in cnn_p.columns else cnn_p['y_prob'].values
    hyb_yt = hyb_p['y_true'].values
    hyb_yp = hyb_p['y_prob'].values
    
    if cnn_yt is not None:
        fpr1,tpr1,_ = roc_curve(cnn_yt, cnn_yp)
        ax1.plot(fpr1, tpr1, label=f'CNN AUC={roc_auc_score(cnn_yt,cnn_yp):.3f}')
        p1,r1,_ = precision_recall_curve(cnn_yt, cnn_yp)
        ax2.plot(r1, p1, label=f'CNN AP={average_precision_score(cnn_yt,cnn_yp):.3f}')
    
    fpr2,tpr2,_ = roc_curve(hyb_yt, hyb_yp)
    ax1.plot(fpr2, tpr2, label=f'Hybrid AUC={roc_auc_score(hyb_yt,hyb_yp):.3f}')
    ax1.plot([0,1],[0,1],'k--'); ax1.legend(); ax1.set_title('ROC Comparison')
    
    p2,r2,_ = precision_recall_curve(hyb_yt, hyb_yp)
    ax2.plot(r2, p2, label=f'Hybrid AP={average_precision_score(hyb_yt,hyb_yp):.3f}')
    ax2.legend(); ax2.set_title('PR Comparison')
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / '13_roc_pr_comparison.png', dpi=100)
    plt.close()

print('Plots saved.')


Plots saved.


## Decision and Handoff


In [5]:
# Decision logic
if len(models) == 2:
    cnn = models.get('LR-CNN-only (lighting-robust)', models.get('CNN-only', {}))
    hyb = models['Hybrid-LR']
    recall_improved = hyb.get('diabetes_recall',0) >= cnn.get('diabetes_recall',0) - 0.01
    spec_ok = hyb.get('specificity',0) >= cnn.get('specificity',0) - 0.03
    roc_ok = hyb.get('roc_auc',0) >= cnn.get('roc_auc',0) - 0.01
    
    if recall_improved and spec_ok and roc_ok:
        decision = 'Hybrid model is at least as good as Lighting-robust CNN-only baseline. Consider adopting hybrid.'
    else:
        decision = 'Hybrid model did NOT clearly improve over Lighting-robust CNN-only baseline. CNN-only remains the safer choice.'
    
    # Check CI overlap
    ci_note = ''
    if HYBRID_CI.exists():
        ci = pd.read_csv(HYBRID_CI)
        ci_note = 'Bootstrap CIs available. Check overlap before claiming improvement.'
else:
    decision = 'Insufficient data for comparison.'
    ci_note = ''

report = f"""IMAGE BRANCH COMPARISON REPORT\n\n{df_comp.to_string(index=False)}\n\nDECISION: {decision}\n{ci_note}\n\nIMPORTANT:\n- Do not claim improvement if CIs overlap heavily.\n- This comparison uses the same locked test set.\n- No threshold tuning on test was performed.\n"""

with open(OUTPUT_DIR / '13_model_comparison_report.txt', 'w') as f:
    f.write(report)
with open(OUTPUT_DIR / '13_decision_recommendation.txt', 'w') as f:
    f.write(decision + '\n' + ci_note)

handoff = f"""FILE 13 HANDOFF\nStatus: PASS\nModels compared: {list(models.keys())}\n{decision}\n{ci_note}\n"""
with open(OUTPUT_DIR / '13_handoff_summary.txt', 'w') as f:
    f.write(handoff)
print(handoff)


FILE 13 HANDOFF
Status: PASS
Models compared: ['LR-CNN-only (lighting-robust)', 'Hybrid-LR']
Hybrid model is at least as good as Lighting-robust CNN-only baseline. Consider adopting hybrid.
Bootstrap CIs available. Check overlap before claiming improvement.

